In [27]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [28]:
import os
import sys

root_path = os.path.abspath("../..")
sys.path.append(root_path)

from algorithm.MADDPG import MADDPGTrainer, MADDPGTester
from network_env.network_env_v4 import NetworkEnvV4

### Directories

In [29]:
RESOURCE_PATH = "./configs/resource_config.json"
LOG_PATH = "./results"

### Configurations

In [30]:

frames_per_batch = 100
n_iter = 100
min_replay_size = 1000
memory_size = 100000
n_optimizer_steps = 100
train_batch_size = 128
actor_lr = 1e-4
critic_lr = 1e-4
max_grad_norm = 1.0
gamma = 0.99
polyak_tau = 0.005

critic_configs = {
        "num_cells":256,
        "depth":2,
        "share_parameter": True,
        "centralized_critic": True
    }

actor_configs = {
        "num_cells":256,
        "depth":2,
        "share_parameter":True
    }

MAX_QUEUE_LENGTH = 5 #2 times capacity
PENALTY = -1
ALPHA = 1.0
BETA = 100.0


### Experiment 1

- Single slice
- Constant Demand = 0.5
- (lambda, rho) = (0.5,0.5), (0.1,0.9), (0.9,0.1)

In [31]:
n_agent = 1
test_demand = 0.5
latency_pref = [0.5, 0.1, 0.9]
energy_pref = [0.5, 0.9, 0.1]


In [32]:
for idx, (lambda_, rho_) in enumerate(zip(latency_pref, energy_pref)):
    print(f'idx={idx}, lambda = {lambda_}, rho = {rho_}')
    
    log_path = os.path.join(LOG_PATH,f"1.{idx}")
    
    train_env = NetworkEnvV4(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=None,
        log_path= os.path.join(log_path,'train'),
        test_demand=test_demand,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[lambda_],
        energy_preference=[rho_]
    )

    test_env = NetworkEnvV4(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=None,
        log_path=os.path.join(log_path,'test'),
        test_demand=test_demand,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[lambda_],
        energy_preference=[rho_]
    )

    trainer = MADDPGTrainer(
        environment=train_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=n_iter,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        save_path=log_path,
        seed = 0
    )

    tester = MADDPGTester(
        environment=test_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=10,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        load_path=log_path
    )
    
    trainer.train()
    tester.test()
    train_env.save_statistics()
    test_env.save_statistics()

idx=0, lambda = 0.5, rho = 0.5


2026-06-05 00:04:30,136 [torchrl][INFO]    Initialized LazyMemmapStorage with torch.Size([100000]) shape [END]


slice_r: 0.147:   1%|          | 1/100 [02:06<3:28:28, 126.35s/it]
/home/tngo/MARL4NetworkSlicing/algorithm/MADDPG.py:419: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  prin

Saved trained policy weights for group 'slice' to ./results/1.0/maddpg_actor_slice.pt
Successfully loaded weights for group 'slice' from ./results/1.0/maddpg_actor_slice.pt
Starting Deterministic Evaluation...


100%|██████████| 10/10 [00:04<00:00,  2.34it/s]


...

2026-06-05 00:08:53,884 [torchrl][INFO]    Initialized LazyMemmapStorage with torch.Size([100000]) shape [END]


slice_r: 0.172:   1%|          | 1/100 [02:14<3:41:38, 134.33s/it]
/home/tngo/MARL4NetworkSlicing/algorithm/MADDPG.py:419: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  prin

Saved trained policy weights for group 'slice' to ./results/1.2/maddpg_actor_slice.pt
Successfully loaded weights for group 'slice' from ./results/1.2/maddpg_actor_slice.pt
Starting Deterministic Evaluation...


100%|██████████| 10/10 [00:04<00:00,  2.31it/s]


[save_statistics] Saving recorder statistics...
[Recorder.save_result] Saved 9997 rewards, 9997 latencies, 9997 energies to results/1.2/train/slice_0
[save_statistics] Saving recorder statistics...
[Recorder.save_result] Saved 996 rewards, 996 latencies, 996 energies to results/1.2/test/slice_0


### Experiment 2

- Single slice
- Constant Demand = 0.3,0.5,0.7
- (lambda, rho) = (0.5,0.5)

In [33]:
n_agent = 1
test_demand = [0.3,0.5,0.7]
latency_pref = 0.5
energy_pref = 0.5

In [34]:
for idx, demand_ in enumerate(test_demand):
    print(f'idx={idx}, demand = {demand_}')
    
    log_path = os.path.join(LOG_PATH,f"2.{idx}")
    
    train_env = NetworkEnvV4(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=None,
        log_path= os.path.join(log_path,'train'),
        test_demand=demand_,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[latency_pref],
        energy_preference=[energy_pref]
    )

    test_env = NetworkEnvV4(
        n_slices=n_agent,
        resource_path=RESOURCE_PATH,
        traffic_path=None,
        log_path=os.path.join(log_path,'test'),
        test_demand=demand_,
        max_queue_length=MAX_QUEUE_LENGTH,
        penalty_reward=PENALTY,
        latency_preference=[latency_pref],
        energy_preference=[energy_pref]
    )

    trainer = MADDPGTrainer(
        environment=train_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=n_iter,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        save_path=log_path,
        seed = 0
    )

    tester = MADDPGTester(
        environment=test_env,
        n_agent=n_agent,
        frames_per_batch=frames_per_batch,
        n_iter=10,
        min_replay_size=min_replay_size,
        memory_size=memory_size,
        n_optimizer_steps= n_optimizer_steps,
        train_batch_size=train_batch_size,
        actor_lr=actor_lr,
        critic_lr=critic_lr,
        max_grad_norm=max_grad_norm,
        polyak_tau=polyak_tau,
        gamma=gamma,
        critic_configs=critic_configs,
        actor_configs=actor_configs,
        load_path=log_path
    )
    
    trainer.train()
    tester.test()
    train_env.save_statistics()
    test_env.save_statistics()

/home/tngo/MARL4NetworkSlicing/.venv/lib/python3.12/site-packages/torchrl/envs/libs/pettingzoo.py:281: UserWarning: PettingZoo in TorchRL is tested using version == 1.24.3 , If you are using a different version and are experiencing compatibility issues,please raise an issue in the TorchRL github.
  warnings.warn(


idx=0, demand = 0.3


2026-06-05 00:11:13,092 [torchrl][INFO]    Initialized LazyMemmapStorage with torch.Size([100000]) shape [END]


slice_r: 0.555:   1%|          | 1/100 [02:06<3:28:37, 126.44s/it]
/home/tngo/MARL4NetworkSlicing/algorithm/MADDPG.py:419: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  prin

Saved trained policy weights for group 'slice' to ./results/2.0/maddpg_actor_slice.pt
Successfully loaded weights for group 'slice' from ./results/2.0/maddpg_actor_slice.pt
Starting Deterministic Evaluation...


...

2026-06-05 00:15:37,480 [torchrl][INFO]    Initialized LazyMemmapStorage with torch.Size([100000]) shape [END]


slice_r: -0.735:   1%|          | 1/100 [02:10<3:35:59, 130.91s/it]
/home/tngo/MARL4NetworkSlicing/algorithm/MADDPG.py:419: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pri

Saved trained policy weights for group 'slice' to ./results/2.2/maddpg_actor_slice.pt
Successfully loaded weights for group 'slice' from ./results/2.2/maddpg_actor_slice.pt
Starting Deterministic Evaluation...


100%|██████████| 10/10 [00:06<00:00,  1.57it/s]


[save_statistics] Saving recorder statistics...
[Recorder.save_result] Saved 9999 rewards, 9999 latencies, 9999 energies to results/2.2/train/slice_0
[save_statistics] Saving recorder statistics...
[Recorder.save_result] Saved 999 rewards, 999 latencies, 999 energies to results/2.2/test/slice_0


In [35]:
import numpy as np
f = lambda utilization : 43.4779 * np.log(100 * utilization) + 226.8324 if np.log(100 * utilization) > 0 else 226.8324